## First run this in dsmlp 
launch-sp26-cuda128.sh -l gpu-class=medium -W CSE151B_SP26_A00 -g 1 -c 8 -m 32 -v a30
##
And check you have enough space to run the model from root dir

# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via Transformer (INT8 quantized)
4. NOTE: DO NOT RUN ON vLLM. The team has notorious trouble with vLLM. ONLY RUN WITH Transformer.
5. Scoring responses against ground-truth answers
6. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Import modules and packages

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [ ]:
#you might need to change DATA_PATH, OUTPUT_PATH
import json
import os

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
BASE_MODEL_ID = MODEL_ID
GPU_ID      = "0"                    
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_SOURCE_LENGTH = 384
MAX_TARGET_LENGTH = 128
MAX_SEQ_LENGTH = MAX_SOURCE_LENGTH + MAX_TARGET_LENGTH
INFERENCE_MAX_SEQ_LENGTH = 2048
MAX_TOKENS = MAX_TARGET_LENGTH
ADAPTER_PATH = "results/qlo_ra_adapter"
MERGED_MODEL_PATH = "results/qlo_ra_merged"
TRAINING_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
MAX_STEPS = 100
LORA_RANK = 64
LORA_ALPHA = 16

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import re
import sys
from pathlib import Path
from typing import Optional
from fractions import Fraction

import gc
import torch

from datasets import Dataset
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from trl import SFTTrainer
from vllm import LLM, SamplingParams

from tqdm import tqdm
from collections import Counter

In [2]:
#check what gpu you have and gpu id number
!nvidia-smi

Sat May 30 16:39:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.211.01             Driver Version: 570.211.01     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L40S                    Off |   00000000:E1:00.0 Off |                    0 |
| N/A   35C    P8             36W /  350W |       0MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [3]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

### Few-shot examples are optional

`build_prompt()` returns the appropriate `(system, user)` pair for each item, and you can turn examples on only when you want longer prompts.

In [ ]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. "
    "Give EXACT answers where possible: fractions or exact expressions like (1/2)^(36/31), not decimals. "
    "If decimals are required, use as many significant figures as possible, never round to fewer. "
    "Keep ALL intermediate calculations to as many significant figures as possible to avoid rounding errors. "
    "COUNT how many values the question asks for and put ALL of them in ONE \\boxed{}. "
    "For example, if the answer is 3.14159 and 2.71828, write \\boxed{3.14159, 2.71828}. "
    "NEVER put intermediate results in \\boxed{}. "
    "Only one \\boxed{} in your entire response, at the very end."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Choose the correct option. "
    "Your final answer must be only \\boxed{X} where X is the option letter. "
    "Do not second-guess your answer. "
    "Only one \\boxed{} in your response."
)

EXAMPLES = """Example 1 (MCQ)
Q: Let $f(x) = x^2 - 2x + 1$. What is the value of $f(3)$?
A. 2
B. 4
C. 6
D. 8
Answer: The function is $f(x) = x^2 - 2x + 1$.
We need to evaluate $f(3)$.
$f(3) = 3^2 - 2(3) + 1 = 9 - 6 + 1 = 4$.
The correct option is B.
\\boxed{B}

Example 2 (Free-form)
Q: A farm has chickens and rabbits. There are 35 heads and 94 feet in total. How many chickens and how many rabbits are on the farm?
Solution: Let c be the number of chickens and r be the number of rabbits.
Each animal has 1 head, so c + r = 35.
Chickens have 2 feet and rabbits have 4 feet, so 2c + 4r = 94.
From the first equation, c = 35 - r.
Substitute this into the second equation:
2(35 - r) + 4r = 94
70 - 2r + 4r = 94
2r = 24
r = 12
Now find c:
c = 35 - 12 = 23.
So there are 23 chickens and 12 rabbits. The question asks for the number of chickens and rabbits.
Final answer: \\boxed{23, 12}
"""


def build_prompt(question: str, options: Optional[list], include_examples: bool = False) -> tuple[str, str]:
    examples_text = EXAMPLES if include_examples else ""
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        user_parts = []
        if examples_text:
            user_parts.append(examples_text)
        user_parts.append(f"{question}\n\nOptions:\n{opts_text}")
        return SYSTEM_PROMPT_MCQ, "\n\n".join(user_parts)
    user_parts = []
    if examples_text:
        user_parts.append(examples_text)
    user_parts.append(question)
    return SYSTEM_PROMPT_MATH, "\n\n".join(user_parts)


def extract_last_boxed(text: str) -> Optional[str]:
    """Extract last \\boxed{} content, handling nested braces like \\frac{5}{8}"""
    results = []
    i = 0
    while i < len(text):
        if text[i:i+7] == r'\boxed{':
            depth = 0
            start = i + 7
            j = start
            while j < len(text):
                if text[j] == '{':
                    depth += 1
                elif text[j] == '}':
                    if depth == 0:
                        results.append(text[start:j])
                        break
                    depth -= 1
                j += 1
        i += 1
    return results[-1].strip() if results else None


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ user prompt (first 200 chars) ──
Example 1 (MCQ)
Q: Let $f(x) = x^2 - 2x + 1$. What is the value of $f(3)$?
A. 2
B. 4
C. 6
D. 8
Answer: The function is $f(x) = x^2 - 2x + 1$.
We need to evaluate $f(3)$.
$f(3) = 3^2 - 2(3) + 1 = 9 - 6 ...\n
── Free-form user prompt (first 200 chars) ──
Example 1 (MCQ)
Q: Let $f(x) = x^2 - 2x + 1$. What is the value of $f(3)$?
A. 2
B. 4
C. 6
D. 8
Answer: The function is $f(x) = x^2 - 2x + 1$.
We need to evaluate $f(3)$.
$f(3) = 3^2 - 2(3) + 1 = 9 - 6 ...\n


## 5. qLoRA Fine-Tuning

We fine-tune **Qwen3-4B-Thinking-2507** with 4-bit QLoRA using the requested settings, tuned for an L40S:

- Effective batch size: 16
- Learning rate: `2e-4`
- Max steps: `10000`
- Source length: `384`
- Target length: `128`
- LoRA rank: `64`
- LoRA alpha: `16`

The notebook trains adapters on the public set, merges them into a standalone model, and then uses that merged model for inference.

In [ ]:
# qLoRA training configuration

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True, fix_mistral_regex=True)
tokenizer.pad_token = tokenizer.eos_token


def create_sft_dataset(items):
    formatted_texts = []
    for item in items:
        system, user = build_prompt(item["question"], item.get("options"), include_examples=False)

        gold_answer = item["answer"]
        if isinstance(gold_answer, list):
            gold_text = ", ".join(str(x) for x in gold_answer)
        else:
            gold_text = str(gold_answer)

        assistant_response = f"\\boxed{{{gold_text}}}"

        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
            {"role": "assistant", "content": assistant_response},
        ]

        formatted_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        formatted_texts.append({"text": formatted_text})

    return Dataset.from_list(formatted_texts)


sft_dataset = create_sft_dataset(data)
print(f"Loaded and formatted {len(sft_dataset)} samples for qLoRA.")
print("\n── qLoRA sample ──")
print(sft_dataset[0]["text"][:800])

# 4-bit quantization for QLoRA
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("Loading base model in 4-bit for qLoRA training...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=quantization_config,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False
model.gradient_checkpointing_enable()

peft_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

training_args = TrainingArguments(
    output_dir=ADAPTER_PATH,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=TRAINING_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=25,
    save_steps=500,
    save_total_limit=2,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    bf16=True,
    fp16=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=sft_dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    processing_class=tokenizer,
    args=training_args,
)

print("Starting qLoRA training...")
trainer.train()
trainer.save_model(ADAPTER_PATH)
print(f"Training complete. LoRA adapters saved to {ADAPTER_PATH}")

# Merge adapters into a standalone model for inference
print("Merging LoRA adapters into a standalone model...")
del trainer
del model
gc.collect()
torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    dtype=torch.bfloat16,
    device_map="cpu",
    trust_remote_code=True,
)

peft_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
merged_model = peft_model.merge_and_unload()

print(f"Saving merged model to {MERGED_MODEL_PATH}...")
merged_model.save_pretrained(MERGED_MODEL_PATH, safe_serialization=True)
tokenizer.save_pretrained(MERGED_MODEL_PATH)

# Release the training-time objects before inference
try:
    del base_model
except NameError:
    pass
try:
    del peft_model
except NameError:
    pass
try:
    del merged_model
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()

print(f"Loading fine-tuned model from {MERGED_MODEL_PATH} with vLLM...")
llm = LLM(
    model=MERGED_MODEL_PATH,
    dtype="bfloat16",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.8,
    max_model_len=INFERENCE_MAX_SEQ_LENGTH,
    trust_remote_code=True,
    max_num_seqs=32,
    max_num_batched_tokens=INFERENCE_MAX_SEQ_LENGTH * 32,
)

sampling_params = SamplingParams(
    n=4,
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
    logprobs=1,
)

print("Fine-tuned model loaded for inference.")

INFO 05-30 16:39:58 [utils.py:233] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'max_model_len': 40960, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.8, 'max_num_batched_tokens': 40960, 'max_num_seqs': 32, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-4B-Thinking-2507'}


INFO 05-30 16:39:58 [model.py:549] Resolved architecture: Qwen3ForCausalLM


INFO 05-30 16:39:58 [model.py:1678] Using max model len 40960


INFO 05-30 16:39:58 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=40960.


INFO 05-30 16:39:58 [vllm.py:790] Asynchronous scheduling is enabled.


...

(EngineCore pid=5461) 

INFO 05-30 16:40:24 [gpu_worker.py:597] CUDA graph pool memory: 0.13 GiB (actual), 0.14 GiB (estimated), difference: 0.01 GiB (8.8%).


(EngineCore pid=5461) 

INFO 05-30 16:40:24 [core.py:283] init engine (profile, create kv cache, warmup model) took 17.73 seconds


Model loaded.


## 6. Generate Responses with the Fine-Tuned Model

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
The prompts now use the shorter qLoRA-friendly form, and vLLM handles batching and scheduling internally.

In [ ]:
# Build prompts for the first 20 entries
prompts = []
for item in data[:20]:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    prompts.append(prompt_text)

# Generate
print(f"Generating responses for {len(prompts)} questions...")
outputs = llm.generate(prompts, sampling_params=sampling_params)

def get_majority_voted_response(output):
    """
    Extract the boxed answer from each generation, find the majority vote,
    and return the full text of the first generation that produced that answer.
    This is a self-consistency method.
    """
    def clean_text(text):
        return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()

    candidates = []
    for o in output.outputs:
        cleaned = clean_text(o.text)
        boxed_answer = extract_last_boxed(cleaned)
        if boxed_answer is not None:
            candidates.append((boxed_answer, o.text.strip()))

    if not candidates:
        # Fallback to the highest logprob response if no boxed answers are found
        def score(o):
            if not o.logprobs:
                return float('-inf')
            total = sum(max(v.logprob for v in step.values()) for step in o.logprobs)
            return total
        best_by_prob = max(output.outputs, key=score)
        return best_by_prob.text.strip()

    answer_counts = Counter(ans for ans, _ in candidates)
    most_common_answer = answer_counts.most_common(1)[0][0]

    # Return the full text of the first response that gave the majority answer
    for ans, txt in candidates:
        if ans == most_common_answer:
            return txt

    return candidates[0][1]

# Using majority voting (self-consistency) instead of picking by log probability
responses = [get_majority_voted_response(out) for out in outputs]

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

Generating responses for 20 questions...


Rendering prompts:   0%|          | 0/20 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/60 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


── Response 0 (id=0) ──
Okay, let's see. The problem is to find the sum of the first 325 positive even whole numbers. Hmm, first I need to remember what the first few even whole numbers are. Positive even whole numbers start at 2, right? So the first one is 2, the second is 4, third is 6, and so on. 

I think there's a formula for the sum of the first n even numbers. Let me recall. The nth even number is 2n, right? Becau ...

── Response 1 (id=1) ──
Okay, let's try to figure out this integral problem. The question is asking for the value of the improper integral from negative infinity to positive infinity of (a^(3/2))/(s² + a²) ds. Hmm, first, I need to recall some standard integrals involving 1/(s² + a²). 

I remember that the integral of 1/(s² + a²) ds from -infty to infty is π/a. Wait, let me confirm that. The antiderivative of 1/(s² + a²) ...

── Response 2 (id=2) ──
Okay, let's tackle this problem step by step. First, I need to remember that this is a Newton's Law of Cooling prob

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [12]:
def strip_thinking(text: str) -> str:
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()
    
def latex_to_numeric(text: str) -> str:
    """Convert \\frac{a}{b} and \\dfrac{a}{b} to decimal strings"""
    def replace_frac(m):
        try:
            num, den = int(m.group(1)), int(m.group(2))
            return str(float(Fraction(num, den)))
        except:
            return m.group(0)
    text = re.sub(r'\\d?frac\{(\d+)\}\{(\d+)\}', replace_frac, text)
    return text
    
def extract_letter(text: str) -> str:
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
    matches = re.findall(r'\\boxed\{([A-Za-z])\}', text)
    if matches:
        return matches[-1].upper()
    matches = re.findall(r'\b([A-Z])\b', text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
## for public test comment out from here
for item, response in tqdm(zip(data, responses), total=len(responses), desc="Scoring"): #chnge data[:20] or [-100:] to check depends on 
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]
    raw_response = response
    response = strip_thinking(response)
    
    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        last_boxed = extract_last_boxed(response)
        pred_converted = latex_to_numeric(last_boxed) if last_boxed is not None else ""
        pred = f"\\\\boxed{{{pred_converted}}}" if pred_converted else response
        try:
            correct = judger.auto_judge(
                pred=pred,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": raw_response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")


#### for private set comment out
# results = []
# for item, response in zip(data, responses):
#     results.append({
#         "id":       item.get("id"),
#         "response": response,  # raw full response for submission
#     })

# print(f"Collected {len(results)} results")



Scoring:   0%|          | 0/20 [00:00<?, ?it/s]


Scoring:  15%|█▌        | 3/20 [00:00<00:01, 10.97it/s]


Scoring:  30%|███       | 6/20 [00:00<00:00, 15.89it/s]


Scoring:  65%|██████▌   | 13/20 [00:00<00:00, 30.14it/s]


Scoring: 100%|██████████| 20/20 [00:00<00:00, 32.13it/s]

Scoring complete. 20 results.


In [13]:
for r in results[:10]:
    predicted = extract_last_boxed(r["response"])
    print(f"\nid={r['id']}")
    print(f"  Gold:      {r['gold']}")
    print(f"  Predicted: {predicted}")
    print(f"  Correct:   {r['correct']}")


id=0
  Gold:      ['325*(1+325)']
  Predicted: None
  Correct:   True

id=1
  Gold:      F
  Predicted: None
  Correct:   False

id=2
  Gold:      ['143.224229233795', '2.32624773420025']
  Predicted: None
  Correct:   False

id=3
  Gold:      ['5/8']
  Predicted: None
  Correct:   True

id=4
  Gold:      C
  Predicted: None
  Correct:   True

id=5
  Gold:      ['62.7777777777778', '335.927777777778', '604.67']
  Predicted: None
  Correct:   True

id=6
  Gold:      ['G', 'B']
  Predicted: None
  Correct:   True

id=7
  Gold:      ['1.44444444444444']
  Predicted: None
  Correct:   True

id=8
  Gold:      ['(1/2)^[(1999-1963)/31]']
  Predicted: None
  Correct:   True

id=9
  Gold:      A
  Predicted: None
  Correct:   True


## 8. Summary

Print accuracy broken down by question type.

In [14]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

EVALUATION RESULTS
  MCQ        :    8 /    9  (88.89%)
  Free-form  :    8 /   11  (72.73%)
  Overall    :   16 /   20  (80.00%)


## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [15]:
SAVE_EVAL = False   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 20 records to results/starter_results.jsonl


In [11]:
# Save submission CSV
import csv
submission_path = Path("results/submission.csv")

with open(submission_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f, quoting=csv.QUOTE_ALL)
    writer.writerow(["id", "response"])
    for r in results:
        writer.writerow([r["id"], r["response"]])

print(f"Saved submission CSV to {submission_path}")

# Sanity check
import pandas as pd
df = pd.read_csv(submission_path)
print(f"Shape: {df.shape}")
print(df.head(3))

Saved submission CSV to results/submission.csv


Shape: (20, 2)
   id                                           response
0   0  Okay, let's see. The problem is to find the su...
1   1  Okay, let's try to figure out this integral pr...
2   2  Okay, let's tackle this problem step by step. ...
